In [ ]:
!pip install transformers datasets accelerate scikit-learn

In [5]:
import os
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

/Users/diogoazevedo/miniconda3/envs/class6dl/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
train_df = pd.read_csv("../data/dataset_treino.csv")
val_df = pd.read_csv("../data/dataset_validacao.csv")
test_df = pd.read_csv("../data/dataset_teste.csv")

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()

(10880, 3) (2332, 3) (2332, 3)


,Text,source_name,source_code
0,Shares will be acquired in accordance with sec...,human,0
1,lmao just saw a funny meme rn,meta,3
2,As the hit TV show Downton Abbey continues to ...,meta,3
3,The building will house product development an...,human,0
4,Nigella Lawson has finally broken her silence ...,human,0


In [7]:
def clean_text(text):
    text = str(text).replace("\n", " ").replace("\t", " ")
    text = " ".join(text.split())
    return text

train_df["Text"] = train_df["Text"].apply(clean_text)
val_df["Text"] = val_df["Text"].apply(clean_text)
test_df["Text"] = test_df["Text"].apply(clean_text)

In [ ]:
USE_SHORT_TEXT = True
MAX_CHARS = 180

if USE_SHORT_TEXT:
    train_df["Text"] = train_df["Text"].astype(str).str.slice(0, MAX_CHARS)
    val_df["Text"] = val_df["Text"].astype(str).str.slice(0, MAX_CHARS)
    test_df["Text"] = test_df["Text"].astype(str).str.slice(0, MAX_CHARS)

train_df["Text"].iloc[0]

In [8]:
label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["source_name"])
val_df["label"] = label_encoder.transform(val_df["source_name"])
test_df["label"] = label_encoder.transform(test_df["source_name"])

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

print(label_encoder.classes_)

['Anthropic' 'google' 'human' 'meta' 'openai']


In [9]:
train_ds = Dataset.from_pandas(train_df[["Text", "label"]])
val_ds = Dataset.from_pandas(val_df[["Text", "label"]])
test_ds = Dataset.from_pandas(test_df[["Text", "label"]])

train_ds

Dataset({
    features: ['Text', 'label'],
    num_rows: 10880
})

In [10]:
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

MAX_LENGTH = 160

def tokenize_function(examples):
    return tokenizer(
        examples["Text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

Map: 100%|██████████| 2332/2332 [00:00<00:00, 6944.36 examples/s]


In [11]:
columns_to_return = ["input_ids", "attention_mask", "label"]

train_ds.set_format(type="torch", columns=columns_to_return)
val_ds.set_format(type="torch", columns=columns_to_return)
test_ds.set_format(type="torch", columns=columns_to_return)

In [12]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

model

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5976.58it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="../models/DistilBERT/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

/Users/diogoazevedo/miniconda3/envs/class6dl/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
pred_output = trainer.predict(test_ds)

test_logits = pred_output.predictions
test_labels = pred_output.label_ids
test_preds = np.argmax(test_logits, axis=1)

print("Test Accuracy:", accuracy_score(test_labels, test_preds))
print("Test Macro F1:", f1_score(test_labels, test_preds, average="macro"))
print(classification_report(
    test_labels,
    test_preds,
    target_names=label_encoder.classes_
))

In [ ]:
save_dir = "../models/DistilBERT/best_model"
os.makedirs(save_dir, exist_ok=True)

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

pd.DataFrame({
    "label_id": list(range(len(label_encoder.classes_))),
    "source_name": label_encoder.classes_
}).to_csv(f"{save_dir}/label_mapping.csv", index=False)

print(f"Modelo guardado em: {save_dir}")

# Submissão

In [ ]:
import pandas as pd
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

In [ ]:
def clean_text(text):
    text = str(text).replace("\n", " ").replace("\t", " ")
    text = " ".join(text.split())
    return text

In [ ]:
input_csv = "../data/subm2.csv"
output_csv = "../subm3/DISTILBERT/subm3.csv"
model_dir = "../models/DistilBERT/best_model"

In [ ]:
df = pd.read_csv(input_csv, sep=";", engine="python")
df.columns = df.columns.str.strip()

print(df.columns)
print(df.head())

df["Text"] = df["Text"].apply(clean_text)

In [ ]:
label_map = pd.read_csv(f"{model_dir}/label_mapping.csv")
id2label = dict(zip(label_map["label_id"], label_map["source_name"]))

tokenizer = DistilBertTokenizerFast.from_pretrained(model_dir)
model = DistilBertForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(id2label)
print(device)

In [ ]:
preds = []
batch_size = 16
MAX_LENGTH = 160

with torch.no_grad():
    for start in range(0, len(df), batch_size):
        batch_texts = df["Text"].iloc[start:start + batch_size].tolist()

        encodings = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encodings = {k: v.to(device) for k, v in encodings.items()}
        outputs = model(**encodings)
        batch_preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        preds.extend(batch_preds)

pred_labels = [id2label[p] for p in preds]
pred_labels[:10]

In [ ]:
submission = pd.DataFrame({
    "ID": df["ID"],
    "Label": pred_labels
})

submission.head()

In [ ]:
submission.to_csv(output_csv, sep=";", index=False)
print(f"Submissão guardada em: {output_csv}")